# Explore the Tension Board 2 SQLite database downloaded via boardlib.

To generate data/tension.db, run the following commands:
- pip install boardlib
- boardlib database tension tension.db

This database should include data for both TB1 and TB2

In [160]:
import sqlite3
from pathlib import Path
import re

In [161]:
DB_PATH = Path("../data/tension.db")

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()

## 1. List all tables

In [162]:
cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
tables = [row[0] for row in cur.fetchall()]
print(f"Tables ({len(tables)}):")
for t in tables:
    count = cur.execute(f"SELECT COUNT(*) FROM [{t}]").fetchone()[0]
    print(f"  {t:40s} {count:>8} rows")

Tables (31):
  android_metadata                                1 rows
  ascents                                         0 rows
  attempts                                       38 rows
  beta_links                                   9500 rows
  bids                                            0 rows
  circuits                                        0 rows
  circuits_climbs                                 0 rows
  climb_cache_fields                          90670 rows
  climb_random_positions                          0 rows
  climb_stats                                147046 rows
  climbs                                     128762 rows
  difficulty_grades                              39 rows
  holes                                         967 rows
  kits                                           25 rows
  layouts                                         3 rows
  leds                                         3388 rows
  placement_roles                                 8 rows
  placements      

## 2. Schema for each table

In [163]:
for t in tables:
    sql = cur.execute(
        "SELECT sql FROM sqlite_master WHERE name=?", (t,)
    ).fetchone()[0]
    print(f"\n{sql};")



CREATE TABLE android_metadata (locale TEXT);

CREATE TABLE ascents (
    uuid TEXT NOT NULL PRIMARY KEY,
    climb_uuid TEXT NOT NULL,
    angle INT UNSIGNED NOT NULL,
    is_mirror BOOLEAN NOT NULL,
    user_id INT UNSIGNED NOT NULL,
    attempt_id INT UNSIGNED NOT NULL,
    bid_count INT UNSIGNED NOT NULL DEFAULT 1,
    quality INT UNSIGNED NOT NULL,
    difficulty INT UNSIGNED NOT NULL,
    is_benchmark INT UNSIGNED NOT NULL DEFAULT 0, -- boolean
    comment TEXT NOT NULL DEFAULT '',
    climbed_at TEXT NOT NULL,
    created_at TEXT NULL DEFAULT NULL,
    FOREIGN KEY (climb_uuid) REFERENCES climbs(uuid) ON UPDATE CASCADE ON DELETE RESTRICT,
    FOREIGN KEY (user_id) REFERENCES users(id) ON UPDATE CASCADE ON DELETE CASCADE,
    FOREIGN KEY (attempt_id) REFERENCES attempts(id) ON UPDATE CASCADE ON DELETE RESTRICT,
    FOREIGN KEY (difficulty) REFERENCES difficulty_grades(difficulty) ON UPDATE CASCADE ON DELETE RESTRICT
);

CREATE TABLE attempts (
    id INT UNSIGNED NOT NULL PRIMARY 

## 3. Explore board grid positions (holes)

In [164]:
for row in cur.execute("SELECT * FROM holes LIMIT 10"):
    print(dict(row))

# x, y ranges
cur.execute("SELECT MIN(x), MAX(x), MIN(y), MAX(y), COUNT(*) FROM holes")
xmin, xmax, ymin, ymax, n = cur.fetchone()
print(f"\nholes: {n} total, x=[{xmin}, {xmax}], y=[{ymin}, {ymax}]")

{'id': 1, 'product_id': 4, 'name': 'A,18', 'x': 8, 'y': 152, 'mirrored_hole_id': 11, 'mirror_group': 0}
{'id': 2, 'product_id': 4, 'name': 'B,18', 'x': 16, 'y': 152, 'mirrored_hole_id': 10, 'mirror_group': 0}
{'id': 3, 'product_id': 4, 'name': 'C,18', 'x': 24, 'y': 152, 'mirrored_hole_id': 9, 'mirror_group': 0}
{'id': 4, 'product_id': 4, 'name': 'D,18', 'x': 32, 'y': 152, 'mirrored_hole_id': 8, 'mirror_group': 0}
{'id': 5, 'product_id': 4, 'name': 'E,18', 'x': 40, 'y': 152, 'mirrored_hole_id': 7, 'mirror_group': 0}
{'id': 6, 'product_id': 4, 'name': 'F,18', 'x': 48, 'y': 152, 'mirrored_hole_id': 6, 'mirror_group': 0}
{'id': 7, 'product_id': 4, 'name': 'G,18', 'x': 56, 'y': 152, 'mirrored_hole_id': 5, 'mirror_group': 0}
{'id': 8, 'product_id': 4, 'name': 'H,18', 'x': 64, 'y': 152, 'mirrored_hole_id': 4, 'mirror_group': 0}
{'id': 9, 'product_id': 4, 'name': 'I,18', 'x': 72, 'y': 152, 'mirrored_hole_id': 3, 'mirror_group': 0}
{'id': 10, 'product_id': 4, 'name': 'J,18', 'x': 80, 'y': 152, 

## 4. Hold placements

In [165]:
for row in cur.execute("SELECT * FROM placements LIMIT 10"):
    print(dict(row))

{'id': 1, 'layout_id': 9, 'hole_id': 2, 'set_id': 8, 'default_placement_role_id': 3}
{'id': 2, 'layout_id': 9, 'hole_id': 10, 'set_id': 8, 'default_placement_role_id': 3}
{'id': 3, 'layout_id': 9, 'hole_id': 317, 'set_id': 8, 'default_placement_role_id': 1}
{'id': 4, 'layout_id': 9, 'hole_id': 325, 'set_id': 8, 'default_placement_role_id': 1}
{'id': 5, 'layout_id': 9, 'hole_id': 320, 'set_id': 8, 'default_placement_role_id': 1}
{'id': 6, 'layout_id': 9, 'hole_id': 322, 'set_id': 8, 'default_placement_role_id': 1}
{'id': 7, 'layout_id': 9, 'hole_id': 321, 'set_id': 8, 'default_placement_role_id': 1}
{'id': 8, 'layout_id': 9, 'hole_id': 43, 'set_id': 8, 'default_placement_role_id': 2}
{'id': 9, 'layout_id': 9, 'hole_id': 53, 'set_id': 8, 'default_placement_role_id': 2}
{'id': 10, 'layout_id': 9, 'hole_id': 86, 'set_id': 8, 'default_placement_role_id': 2}


## 5. Placement roles

In [166]:
for row in cur.execute("SELECT * FROM placement_roles"):
    print(dict(row))

{'id': 1, 'product_id': 4, 'position': 1, 'name': 'start', 'full_name': 'Start', 'led_color': '00FF00', 'screen_color': '00DD00'}
{'id': 3, 'product_id': 4, 'position': 3, 'name': 'finish', 'full_name': 'Finish', 'led_color': 'FF0000', 'screen_color': 'FF0000'}
{'id': 4, 'product_id': 4, 'position': 4, 'name': 'foot', 'full_name': 'Foot Only', 'led_color': 'FF00FF', 'screen_color': 'FF00FF'}
{'id': 5, 'product_id': 5, 'position': 1, 'name': 'start', 'full_name': 'Start', 'led_color': '00FF00', 'screen_color': '00DD00'}
{'id': 7, 'product_id': 5, 'position': 3, 'name': 'finish', 'full_name': 'Finish', 'led_color': 'FF0000', 'screen_color': 'FF0000'}
{'id': 8, 'product_id': 5, 'position': 4, 'name': 'foot', 'full_name': 'Foot Only', 'led_color': 'FF00FF', 'screen_color': 'FF00FF'}
{'id': 2, 'product_id': 4, 'position': 2, 'name': 'middle', 'full_name': 'Middle', 'led_color': '0000FF', 'screen_color': '0066FF'}
{'id': 6, 'product_id': 5, 'position': 2, 'name': 'middle', 'full_name': 'Midd

## 6. LEDs

In [167]:
for row in cur.execute("SELECT * FROM leds LIMIT 10"):
    print(dict(row))

{'id': 1, 'product_size_id': 1, 'hole_id': 379, 'position': 0}
{'id': 2, 'product_size_id': 1, 'hole_id': 389, 'position': 1}
{'id': 3, 'product_size_id': 1, 'hole_id': 378, 'position': 2}
{'id': 4, 'product_size_id': 1, 'hole_id': 388, 'position': 3}
{'id': 5, 'product_size_id': 1, 'hole_id': 377, 'position': 4}
{'id': 6, 'product_size_id': 1, 'hole_id': 387, 'position': 5}
{'id': 7, 'product_size_id': 1, 'hole_id': 376, 'position': 6}
{'id': 8, 'product_size_id': 1, 'hole_id': 386, 'position': 7}
{'id': 9, 'product_size_id': 1, 'hole_id': 375, 'position': 8}
{'id': 10, 'product_size_id': 1, 'hole_id': 385, 'position': 9}


## 7. Climbs

In [168]:
for row in cur.execute("SELECT * FROM climbs LIMIT 5"):
    d = dict(row)
    if 'frames' in d and d['frames'] and len(str(d['frames'])) > 80:
        d['frames'] = str(d['frames']) + '...'
    print(d)

{'uuid': '00163801596af1064d549ad75b684539', 'layout_id': 9, 'setter_id': 33802, 'setter_username': 'vinsewah', 'name': 'Duroxmanie 2.0', 'description': 'No matching', 'hsm': 3, 'edge_left': 8, 'edge_right': 88, 'edge_bottom': 32, 'edge_top': 128, 'angle': None, 'frames_count': 1, 'frames_pace': 0, 'frames': 'p3r4p29r2p59r1p65r2p75r3p89r2p157r4p158r4', 'is_draft': 0, 'is_listed': 1, 'created_at': '2021-02-16 09:13:28.000000', 'is_nomatch': 1}
{'uuid': '001945feb7509ce231c9d8b237082a39', 'layout_id': 9, 'setter_id': 30521, 'setter_username': 'ssssss', 'name': '1 am', 'description': '', 'hsm': 3, 'edge_left': 24, 'edge_right': 72, 'edge_bottom': 56, 'edge_top': 128, 'angle': None, 'frames_count': 1, 'frames_pace': 0, 'frames': 'p18r1p22r1p57r2p70r2p149r3p161r4p162r4', 'is_draft': 0, 'is_listed': 1, 'created_at': '2021-02-13 01:53:03.000000', 'is_nomatch': 0}
{'uuid': '002078ce5b07166d80e87d2cafc94dab', 'layout_id': 9, 'setter_id': 61740, 'setter_username': 'rockindude', 'name': 'test69',

## 8. Products and Product sizes (TB1 vs TB2)

In [169]:
for row in cur.execute("SELECT * FROM products"):
    print(dict(row))

for row in cur.execute("SELECT * FROM product_sizes"):
    print(dict(row))

{'id': 4, 'name': 'Tension Board', 'is_listed': 1, 'password': None, 'min_count_in_frame': 2, 'max_count_in_frame': 35}
{'id': 5, 'name': 'Tension Board 2', 'is_listed': 1, 'password': None, 'min_count_in_frame': 2, 'max_count_in_frame': 35}
{'id': 1, 'product_id': 4, 'edge_left': 0, 'edge_right': 96, 'edge_bottom': 0, 'edge_top': 156, 'name': 'Full Wall', 'description': 'Rows: KB1, KB2, 1-18\nColumns: A-K', 'image_filename': 'product_sizes/1.png', 'position': 0, 'is_listed': 1}
{'id': 2, 'product_id': 4, 'edge_left': 0, 'edge_right': 96, 'edge_bottom': 4, 'edge_top': 156, 'name': 'Half Kickboard', 'description': 'Rows: KB2, 1-18\nColumns: A-K', 'image_filename': 'product_sizes/2.png', 'position': 1, 'is_listed': 1}
{'id': 3, 'product_id': 4, 'edge_left': 0, 'edge_right': 96, 'edge_bottom': 8, 'edge_top': 156, 'name': 'No Kickboard', 'description': 'Rows: 1-18\nColumns: A-K', 'image_filename': 'product_sizes/3-v2.png', 'position': 2, 'is_listed': 1}
{'id': 4, 'product_id': 4, 'edge_lef

## 9. Layouts

In [170]:
for row in cur.execute("SELECT * FROM layouts"):
    print(dict(row))

cur.execute("""
    SELECT l.name, l.id, COUNT(*) as n_placements
    FROM placements p
    JOIN layouts l ON p.layout_id = l.id
    GROUP BY p.layout_id
""")
for row in cur.fetchall():
    print(dict(row))

{'id': 10, 'product_id': 5, 'name': 'Tension Board 2 Mirror', 'instagram_caption': '', 'is_mirrored': 1, 'is_listed': 1, 'password': None, 'created_at': '2022-08-19 14:52:19.570731'}
{'id': 11, 'product_id': 5, 'name': 'Tension Board 2 Spray', 'instagram_caption': '', 'is_mirrored': 0, 'is_listed': 1, 'password': None, 'created_at': '2022-10-26 03:42:45.736011'}
{'id': 9, 'product_id': 4, 'name': 'Original Layout', 'instagram_caption': '', 'is_mirrored': 1, 'is_listed': 1, 'password': None, 'created_at': '2017-01-01 00:45:51.000000'}
{'name': 'Original Layout', 'id': 9, 'n_placements': 303}
{'name': 'Tension Board 2 Mirror', 'id': 10, 'n_placements': 498}
{'name': 'Tension Board 2 Spray', 'id': 11, 'n_placements': 498}


## 10. Parse a sample climb's frames string

Format: p<placement_id>r<role_id>p<placement_id>r<role_id>...

In [171]:
sample = cur.execute(
    "SELECT uuid, name, frames FROM climbs WHERE layout_id IN (10, 11) AND frames IS NOT NULL AND frames != '' LIMIT 1"
).fetchone()
if sample:
    climb_uuid, name, frames = sample
    print(f"Climb: {name} (uuid={climb_uuid})")
    print(f"Raw frames: {frames}")
    
    # Parse p<id>r<role> pairs
    pairs = re.findall(r'p(\d+)r(\d+)', frames)
    print(f"\nParsed {len(pairs)} placements:")
    for placement_id, role_id in pairs:
        row = cur.execute("""
            SELECT p.id as placement_id, p.hole_id, h.x, h.y, h.name as hole_name,
                   pr.full_name as role_name
            FROM placements p
            JOIN holes h ON p.hole_id = h.id
            LEFT JOIN placement_roles pr ON pr.id = ?
            WHERE p.id = ?
        """, (role_id, placement_id)).fetchone()
        if row:
            print(f"  placement={row['placement_id']}, hole={row['hole_name']}, "
                  f"pos=({row['x']}, {row['y']}), role={row['role_name']}")

Climb: Hermes (uuid=0A70BF5C001641D3BBE59E11528A7251)
Raw frames: p466r8p477r6p480r6p485r8p490r6p492r7p498r5p500r6p504r8p507r6p509r6p514r8

Parsed 12 placements:
  placement=466, hole=24,4, pos=(24, 4), role=Foot Only
  placement=477, hole=28,56, pos=(28, 56), role=Middle
  placement=480, hole=28,96, pos=(28, 96), role=Middle
  placement=485, hole=32,28, pos=(32, 28), role=Foot Only
  placement=490, hole=32,116, pos=(32, 116), role=Middle
  placement=492, hole=32,140, pos=(32, 140), role=Finish
  placement=498, hole=40,44, pos=(40, 44), role=Start
  placement=500, hole=40,76, pos=(40, 76), role=Middle
  placement=504, hole=44,16, pos=(44, 16), role=Foot Only
  placement=507, hole=44,104, pos=(44, 104), role=Middle
  placement=509, hole=44,128, pos=(44, 128), role=Middle
  placement=514, hole=48,68, pos=(48, 68), role=Foot Only


## 11. Number of climbs per layout

In [172]:
for row in cur.execute("""
    SELECT l.name, l.id, COUNT(*) as n_climbs
    FROM climbs c JOIN layouts l ON c.layout_id = l.id
    GROUP BY c.layout_id
"""):
    print(dict(row))

{'name': 'Original Layout', 'id': 9, 'n_climbs': 68511}
{'name': 'Tension Board 2 Mirror', 'id': 10, 'n_climbs': 39396}
{'name': 'Tension Board 2 Spray', 'id': 11, 'n_climbs': 20855}


## 12. TB2 coordinate system

In [173]:
cur.execute("""
    SELECT MIN(x), MAX(x), MIN(y), MAX(y), COUNT(*)
    FROM holes WHERE product_id = 5
""")
xmin, xmax, ymin, ymax, n = cur.fetchone()
print(f"TB2 holes: {n} total, x=[{xmin}, {xmax}], y=[{ymin}, {ymax}]")

TB2 holes: 578 total, x=[-64, 64], y=[4, 140]


In [174]:
conn.close()